# Seismic Hazard Application

Run this notebook to reproduce the FQ-IDCVT seismic hazard workflow from modular functions.

## Correction Notes

> `B_EPS_KM = 8.5` (Jayaram & Baker 2009, T=0.1s), not 20.0.
> Correlation formula is $\exp(-3h/b)$, not $\exp(-h/b)$.
> Quantizer sizes are `N = 50 and 200`, not 500. Latitude/longitude axes are preserved for map plots.

In [ ]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "fq").exists():
        sys.path.insert(0, str(candidate))
        break
import numpy as np
import matplotlib.pyplot as plt

from applications.seismic_hazard.geometry import (
    FAULT_TABLE, LAT_PLOT, LON_PLOT, fault_xy, lat_grid, lon_grid,
    to_2d, x_grid, y_grid,
)
from applications.seismic_hazard.correlation import B_EPS_KM, jb2009_rho
from applications.seismic_hazard.gmm import as97_ln_sa
from applications.seismic_hazard.simulation import simulate_maps
from applications.seismic_hazard.fq_idcvt import fq_idcvt
print(B_EPS_KM, jb2009_rho(8.5), np.exp(as97_ln_sa(6.5, 10)))

## Geometry Diagram

Plot fault nodes A-D, site grid, and fault traces.

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(LON_PLOT, LAT_PLOT, s=4, alpha=0.35)
for node, (lat, lon) in FAULT_TABLE.items():
    ax.scatter(lon, lat, color="red"); ax.text(lon, lat, node)
for a,b in [("A","B"),("C","D")]:
    ax.plot([FAULT_TABLE[a][1], FAULT_TABLE[b][1]], [FAULT_TABLE[a][0], FAULT_TABLE[b][0]], color="black")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.set_title("Fault nodes and site grid")
plt.show()

## Simulation and FQ-IDCVT

Use small defaults for an executable smoke run. Increase `N`, `nsim_factor`, and `max_iter` for paper-scale figures.

In [ ]:
rng = np.random.default_rng(42)
X_log = simulate_maps(20, rng)
X_sa = np.exp(X_log)
q_log, weights, info = fq_idcvt(5, np.random.default_rng(43), max_iter=5, nsim_factor=5, npsim_factor=5)
q_sa = np.exp(q_log)
print(X_log.shape, q_sa.shape, weights.sum(), info)

## Figures 2.8-2.14

Use the same helpers to draw surfaces, exceedance maps, and autocorrelation comparisons for the paper-scale `N=50` and `N=200` runs.

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
Z = to_2d((X_sa > 0.3).mean(axis=0))
cs = ax.contourf(LON_PLOT, LAT_PLOT, Z, levels=10)
plt.colorbar(cs, ax=ax, label="P[Sa > 0.3g]")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.set_title("Fig. 2.14 style exceedance map")
plt.show()